# 3.12 Performans: Değerlendirme ve Sorgu

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/12-performance-eval-and-query.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Performance Eval and Query

Önceki bölümlerde gördüğümüz gibi PyData yığınının gücü, NumPy ve Pandas'ın temel işlemleri sezgisel üst düzey sözdizimiyle alt düzey derlenmiş koda itme yeteneğine dayanır: NumPy'de vektörize/yayınlanmış işlemler, Pandas'ta gruplama türü işlemler buna örnektir. Bu soyutlamalar birçok yaygın kullanım için verimli olsa da genelde geçici ara nesneler oluşturmaya dayanır; bu da hesaplama süresi ve bellek kullanımında gereksiz yük yaratabilir.

Bunu gidermek için Pandas, ara dizilerin maliyetli ayrılması olmadan doğrudan C hızında işlemlere erişmeyi sağlayan yöntemler içerir: eval ve query. Bunlar NumExpr paketine dayanır. Bu bölümde kullanımlarını ve ne zaman düşünmeniz gerekebileceğine dair pratik kuralları ele alacağız.

## query ve eval Motivasyonu: Bileşik İfadeler

NumPy ve Pandas'ın hızlı vektörize işlemleri desteklediğini daha önce gördük; örneğin iki dizinin elemanlarını toplarken:


In [ ]:
# timeit_numpy_add.py
import numpy as np
rng = np.random.default_rng(42)
x = rng.random(1000000)
y = rng.random(1000000)
%timeit x + y



2.3 Evrensel Fonksiyonlar bölümünde tartışıldığı gibi bu, Python döngüsü veya comprehension ile toplama yapmaktan çok daha hızlıdır:


In [ ]:
# timeit_loop_add.py
%timeit np.fromiter((xi + yi for xi, yi in zip(x, y)),
                    dtype=x.dtype, count=len(x))



Ancak bileşik ifadeler hesaplanırken bu soyutlama daha az verimli olabilir. Şu ifadeyi düşünün:


In [ ]:
# compound_mask.py
mask = (x > 0.5) & (y < 0.5)



NumPy her alt ifadeyi ayrı değerlendirdiği için bu kabaca şuna eşdeğerdir:


In [ ]:
# compound_tmp.py
tmp1 = (x > 0.5)
tmp2 = (y < 0.5)
mask = tmp1 & tmp2



Başka bir deyişle, her ara adım bellekte açıkça ayrılır. x ve y dizileri çok büyükse bu bellek ve hesaplama yüküne yol açabilir.

NumExpr kütüphanesi bu tür bileşik ifadeyi tam ara diziler ayırmadan eleman eleman hesaplamanıza olanak tanır. NumExpr dokümantasyonu daha fazla ayrıntı içerir; şimdilik kütüphanenin hesaplamak istediğiniz NumPy tarzı ifadeyi veren bir dize kabul ettiğini söylemek yeterlidir:


In [ ]:
# numexpr_eval.py
import numexpr
mask_numexpr = numexpr.evaluate('(x > 0.5) & (y < 0.5)')
np.all(mask == mask_numexpr)



NumExpr ifadeyi mümkün olduğunca geçici dizilerden kaçınarak değerlendirir; bu yüzden özellikle büyük diziler üzerinde uzun hesaplama dizilerinde NumPy'den çok daha verimli olabilir. Burada ele alacağımız Pandas eval ve query araçları kavramsal olarak benzerdir; esasen NumExpr işlevselliğinin Pandas'a özgü sarmalayıcılarıdır.

> **Not**
>

## Verimli İşlemler için pandas.eval

Pandas'taki eval fonksiyonu dize ifadeleri kullanarak DataFrame nesneleri üzerinde verimli hesaplamalar yapar. Örneğin şu veriyi düşünün:


In [ ]:
# eval_dataframes.py
import pandas as pd
nrows, ncols = 100000, 100
df1, df2, df3, df4 = (pd.DataFrame(rng.random((nrows, ncols)))
                      for i in range(4))



Dört DataFrame'in toplamını tipik Pandas yaklaşımıyla yazabiliriz:


In [ ]:
# timeit_df_sum.py
%timeit df1 + df2 + df3 + df4



Aynı sonuç pd.eval ile dize olarak ifade kurularak elde edilebilir:


In [ ]:
# timeit_pd_eval.py
%timeit pd.eval('df1 + df2 + df3 + df4')



Bu ifadenin eval sürümü yaklaşık %50 daha hızlıdır (ve çok daha az bellek kullanır) ve aynı sonucu verir:


In [ ]:
# eval_allclose.py
np.allclose(df1 + df2 + df3 + df4,
            pd.eval('df1 + df2 + df3 + df4'))



pd.eval geniş bir işlem yelpazesini destekler. Göstermek için şu tamsayı verisini kullanacağız:


In [ ]:
# eval_int_data.py
df1, df2, df3, df4, df5 = (pd.DataFrame(rng.integers(0, 1000, (100, 3)))
                           for i in range(5))



#### Aritmetik operatörler


In [ ]:
# eval_arithmetic.py
result1 = -df1 * df2 / (df3 + df4) - df5
result2 = pd.eval('-df1 * df2 / (df3 + df4) - df5')
np.allclose(result1, result2)



#### Karşılaştırma operatörleri


In [ ]:
# eval_comparison.py
result1 = (df1 < df2) & (df2 <= df3) & (df3 != df4)
result2 = pd.eval('df1 < df2 <= df3 != df4')
np.allclose(result1, result2)



#### Bit düzeyi operatörler


In [ ]:
# eval_bitwise.py
result1 = (df1 < 0.5) & (df2 < 0.5) | (df3 < df4)
result2 = pd.eval('(df1 < 0.5) & (df2 < 0.5) | (df3 < df4)')
np.allclose(result1, result2)



Ayrıca Boolean ifadelerde and ve or kullanımını destekler:


In [ ]:
# eval_and_or.py
result3 = pd.eval('(df1 < 0.5) and (df2 < 0.5) or (df3 < df4)')
np.allclose(result1, result3)



#### Nesne öznitelikleri ve indeksler


In [ ]:
# eval_attr_index.py
result1 = df2.T[0] + df3.iloc[1]
result2 = pd.eval('df2.T[0] + df3.iloc[1]')
np.allclose(result1, result2)



#### Diğer işlemler

Fonksiyon çağrıları, koşullu ifadeler, döngüler ve daha karmaşık yapılar şu an pd.eval içinde uygulanmamıştır. Bu tür ifadeleri çalıştırmak için NumExpr kütüphanesinin kendisini kullanabilirsiniz.


In [ ]:
# df_eval_ornek.py
df = pd.DataFrame(rng.random((1000, 3)), columns=['A', 'B', 'C'])
df.head()



Önceki bölümdeki gibi pd.eval ile üç sütunlu ifade hesaplayabiliriz:


In [ ]:
# pd_eval_columns.py
result1 = (df['A'] + df['B']) / (df['C'] - 1)
result2 = pd.eval("(df.A + df.B) / (df.C - 1)")
np.allclose(result1, result2)



DataFrame.eval yöntemi sütunlarla ifadelerin çok daha özlü değerlendirilmesine izin verir:


In [ ]:
# df_eval_columns.py
result3 = df.eval('(A + B) / (C - 1)')
np.allclose(result1, result3)



Burada sütun adlarını değerlendirilen ifade içinde değişken gibi ele aldığımıza ve sonucun istediğimiz gibi olduğuna dikkat edin.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      DataFrame.eval ile sütun adlarını değişken gibi kullanın:
      
        import pandas as pd
import numpy as np
rng = np.random.default_rng(0)
df = pd.DataFrame(rng.random((5, 3)), columns=[&quot;A&quot;, &quot;B&quot;, &quot;C&quot;])
df.eval(&quot;D = (A + B) / C&quot;, inplace=True)
print(df.head())

### DataFrame.eval'de Atama


In [ ]:
# df_head.py
df.head()



df.eval ile diğer sütunlardan hesaplanan yeni bir 'D' sütunu oluşturup atayabiliriz:


In [ ]:
# df_eval_assign.py
df.eval('D = (A + B) / C', inplace=True)
df.head()



Aynı şekilde mevcut herhangi bir sütun değiştirilebilir:


In [ ]:
# df_eval_modify.py
df.eval('D = (A - B) / C', inplace=True)
df.head()



### DataFrame.eval'de Yerel Değişkenler


In [ ]:
# eval_local_var.py
column_mean = df.mean(1)
result1 = df['A'] + column_mean
result2 = df.eval('A + @column_mean')
np.allclose(result1, result2)



Buradaki @ karakteri bir sütun adı değil değişken adı işaretler; sütun ad alanı ile Python nesne ad alanını içeren ifadeleri verimli değerlendirmenizi sağlar. Bu @ yalnızca DataFrame.eval yöntemi tarafından desteklenir; pandas.eval fonksiyonu yalnızca tek (Python) ad alanına erişebildiği için @ desteklemez.

## DataFrame.query Yöntemi

DataFrame'in değerlendirilmiş dizelere dayanan bir başka yöntemi query'dir:


In [ ]:
# query_mask_equiv.py
result1 = df[(df.A < 0.5) & (df.B < 0.5)]
result2 = pd.eval('df[(df.A < 0.5) & (df.B < 0.5)]')
np.allclose(result1, result2)



DataFrame.eval tartışmasındaki örnek gibi bu da DataFrame sütunlarını içeren bir ifadedir. Ancak DataFrame.eval sözdizimiyle ifade edilemez! Bu tür filtreleme için query yöntemini kullanabilirsiniz:


In [ ]:
# df_query.py
result2 = df.query('A < 0.5 and B < 0.5')
np.allclose(result1, result2)



Daha verimli hesaplamanın yanı sıra maskeleme ifadesine kıyasla okunması ve anlaşılması çok daha kolaydır. query yöntemi yerel değişkenler için de @ bayrağını kabul eder:

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      query ile okunabilir filtreleme:
      
        import pandas as pd
import numpy as np
rng = np.random.default_rng(1)
df = pd.DataFrame(rng.random((100, 3)), columns=[&quot;A&quot;, &quot;B&quot;, &quot;C&quot;])
sub = df.query(&quot;A &lt; 0.3 and B &gt; 0.5&quot;)
print(len(sub), &quot;satır seçildi&quot;)


In [ ]:
# query_local_var.py
Cmean = df['C'].mean()
result1 = df[(df.A < Cmean) & (df.B < Cmean)]
result2 = df.query('A < @Cmean and B < @Cmean')
np.allclose(result1, result2)



## Performans: Bu Fonksiyonları Ne Zaman Kullanmalı?

eval ve query kullanılıp kullanılmayacağını değerlendirirken iki faktör vardır: hesaplama süresi ve bellek kullanımı. Bellek kullanımı en öngörülebilir yöndür. Daha önce belirtildiği gibi NumPy dizileri veya Pandas DataFrame'leri içeren her bileşik ifade geçici dizilerin örtük oluşturulmasına yol açar. Örneğin şu:


In [ ]:
# mask_filter.py
x = df[(df.A < 0.5) & (df.B < 0.5)]



kabaca şuna eşdeğerdir:


In [ ]:
# mask_tmp_steps.py
tmp1 = df.A < 0.5
tmp2 = df.B < 0.5
tmp3 = tmp1 & tmp2
x = df[tmp3]



Geçici DataFrame'lerin boyutu kullanılabilir sistem belleğinize (genelde birkaç gigabayt) göre önemliyse eval veya query ifadesi kullanmak iyi bir fikirdir. Dizinizin yaklaşık bayt cinsinden boyutunu şununla kontrol edebilirsiniz:


In [ ]:
# df_nbytes.py
df.values.nbytes



Performans tarafında, sistem belleğinizi doldurmasanız bile eval daha hızlı olabilir. Sorun geçici nesnelerinizin sisteminizdeki L1 veya L2 CPU önbelleği boyutuyla (genelde birkaç megabayt) karşılaştırmasıdır; çok daha büyüklerse eval farklı bellek önbellekleri arasında yavaş değer taşınmasını önleyebilir.

Pratikte geleneksel yöntemler ile eval/query arasındaki hesaplama süresi farkı genelde önemli değildir — hatta küçük dizilerde geleneksel yöntem daha hızlı olabilir! eval/query'nin asıl faydası tasarruf edilen bellek ve bazen daha temiz sözdizimidir.

Burada eval ve query'nin çoğu ayrıntısını ele aldık; daha fazlası için Pandas dokümantasyonuna bakabilirsiniz. Özellikle bu sorguları çalıştırmak için farklı ayrıştırıcılar ve motorlar belirtilebilir; ayrıntılar için dokümantasyondaki "Enhancing Performance" bölümüne bakın.

> **Not**
>

> **Not**
>
